<a href="https://colab.research.google.com/github/alokchoudharyguliya/Transformers/blob/main/Text_Summarization_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip3 install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from spacy.tokenizer import Tokenizer
from sklearn.model_selection import train_test_split
import spacy
import pandas as pd
import numpy as np
import os, re
from nltk.corpus import stopwords
import random
from tqdm import tqdm
import math

In [ ]:
nlp=spacy.load('en_core_web_sm')
tokenizer=Tokenizer(nlp.vocab)

In [ ]:
# Add data from files into dataframes for easier access
def create_dataframe(source_text_path,target_text_path):
  txt_files_source=[file for file in os.listdir(source_text_path) if file.endswith('.txt')]
  txt_files_target=[file for file in os.listdir(target_text_path) if file.endswith('.txt')]
  df=pd.DataFrame(columns=['headlines','text'])
  for source,target in zip(txt_files_source,txt_files_target):
    assert source==target
    source_file_path=os.path.join(source_text_path,source)
    target_file_path=os.path.join(target_text_path,target)

    # Read the content of the file
    with open(source_file_path,'r',encoding='latin-1') as file:
      source_text=file.read()
    with open(target_file_path,'r',encoding='latin-1') as file:
      target_text=file.read()
    df.loc[len(df.index)]=[source_text,target_text]
  return df

In [ ]:
df1=create_dataframe("")

Kaggle Dataset Fetch

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = "pseudophoenix"
os.environ["KAGGLE_KEY"] = "7a2cb2f192719312c0d0e000fbb07af1"


In [ ]:
!kaggle kernels pull kiranraghavendra/text-summarization-transformer-from-scratch

Source code downloaded to /content/text-summarization-transformer-from-scratch.ipynb


In [ ]:
import kagglehub
# Download the specific dataset
base_path = kagglehub.dataset_download("pariza/bbc-news-summary")

100%|██████████| 8.91M/8.91M [00:00<00:00, 164MB/s]

Extracting files...


Extracting all the articles and summaries data

In [ ]:
articles_root=f"{base_path}/BBC News Summary/News Articles"
summaries_root=f"{base_path}/BBC News Summary/Summaries"
df1 = create_dataframe(f"{articles_root}/business",f"{summaries_root}/business")
df2 = create_dataframe(f"{articles_root}/entertainment",f"{summaries_root}/entertainment")
df3 = create_dataframe(f"{articles_root}/politics",f"{summaries_root}/politics")
df4 = create_dataframe(f"{articles_root}/sport",f"{summaries_root}/sport")
df5 = create_dataframe(f"{articles_root}/tech",f"{summaries_root}/tech")

Combining all the df s into one dataframe

In [ ]:
df=pd.concat([df1,df2,df3,df4,df5],ignore_index=True)

In [ ]:
df=df.rename(columns={"headlines":"source_text","text":"summary_text"})
X,y=df['source_text'],df['summary_text']
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)
train_df=pd.DataFrame({'source_text':X_train,'summary_text':X_test})
test_df=pd.DataFrame({'source_text':y_train,'summary_text':y_test})

In [ ]:
contraction_mapping={}

In [ ]:
def process_dataframe(df,tokenizer, text_cleaner_func):
  df['summary_text']=df['summary_text'].apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner_func(x))])
  df['source_text']=df['source_text'].apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner_func(x))])
  return df

In [ ]:
def text_cleaner(text):
  # This is a placeholder for the actual text cleaning logic
  # Based on common text summarization pipelines, this function would typically handle:
  # - Removing special characters, punctuation, and numbers
  # - Removing extra spaces
  # - Lowercasing (though your current code does this separately)
  # - Removing stopwords (if desired)
  # - Expanding contractions (if using the contraction_mapping)
  # For now, it just returns the text as is.
  new_text = text.lower()
  new_text = re.sub(r"[^a-zA-Z0-9 ]", "", new_text) # Remove special characters
  new_text = re.sub(r"\s+", " ", new_text) # Remove extra spaces
  return new_text

In [ ]:
def process_dataframe_text(df, tokenizer, text_cleaner_func):
  df['source_text'] = df['source_text'].apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner_func(x))])
  df['summary_text'] = df['summary_text'].apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner_func(x))])
  return df

In [ ]:
# Tokenize and lowercase text using spacy
train_df = process_dataframe_text(train_df, tokenizer, text_cleaner)
test_df = process_dataframe_text(test_df, tokenizer, text_cleaner)

In [ ]:
def text_cleaner(text):
  return text

In [ ]:
# Tokenize and lowercase text using spacy
# train_df['source_text']=train_df['source_text'].apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner(x))])
# train_df=process_dataframe(train_df,tokenizer,text_cleaner)
# train_df['summary_text']=train_df['summary_text'].apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner(x))])

# Fill NaN values with empty strings before tokenization to prevent TypeError
test_df['source_text']=test_df['source_text'].fillna('').apply(lambda x:[token.text.lower() for token in tokenizer(text_cleaner(x))])
test_df['summary_text']=test_df['summary_text'].fillna('').apply(lambda x: [token.text.lower() for token in tokenizer(text_cleaner(x))])

TypeError: Argument 'string' has incorrect type (expected str, got float)

In [ ]:
train_df['source_text']=train_df['source_text'].apply(lambda x:['_START_']+x+['_END_'])
train_df['summary_text']=train_df['summary_text'].apply(lambda x:['_START_']+x+['_END_'])

test_df['source_text']=test_df['source_text'].apply(lambda x:['_START_']+x+['_END_'])
test_df['summary_text']=test_df['summary_text'].apply(lambda x:['_START_']+x+['_END_'])

TypeError: can only concatenate list (not "str") to list

In [ ]:
train_df.head()

In [ ]:
# Builind vocabularies - each word has an index, note: words sorted in ascending order
all_tokens=train_df['source_text'].tolist()+train_df['summary_text'].tolist()+test_df['source_text'].tolist()+test_df['summary_text'].tolist()
source_vocab={actual_word:idx for idx, (word_num, actual_word) in enumerate(sorted(enumerate(set(token for token in all_tokens for token in token in tokens)),key=lambda x: x[1]))}
target_vocab={actual_wrod: idx for idx, (word_num, actual_word) in enumerate(sorted(enumerate(set(token for token in all_tokens for token in tokens)), key = lambda x: x[1])}

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("using ",device)

using  cpu


In [ ]:
temp=list(sorted(source_vocab.items()))
for word, idx in temp[-5:]:
  print(word, idx)

In [ ]:
# Define a custom dataset class
class CustomDataset(Dataset):
  def __init__(self, source_texts, target_summaries, source_vocab, target_vocab):
    self.source_texts=source_texts
    self.target_summaries=target_summaries
    self.source_vocab=source_vocab
    self.target_vocab=target_vocab

  def __len__(self):
    return len(self.source_texts)

  def __getitem__(self,idx):
    source_text=[self.source_vocab[word] for word in self.source_texts[idx]]
    target_summary=[self.target_vocab[word] for word in self.target_summaries[idx]]
    return torch.tensor(source_text), torch.tensor(target_summary)


In [ ]:
train_dataset=CustomDataset(train_df['source_text'].tolist(),train_df['summary_text'].tolist(),source_vocab, target_vocab)
test_dataset=CustomDataset(test_df['source_text'].tolist(),test_df['summary_text'].tolist(),source_vocab,target_vocab)

NameError: name 'source_vocab' is not defined

In [ ]:
def get_max_seqlen():
  max_length=0
  for index, row in train_df.iterrows():
    # Calculate the length of the current row
    row_length=len(row['source_text'])
    # Update the maximum length if the current row length is greater
    max_length=max(max_length,row_length)
  for index, row in test_df.iterrows():
    row_length=len(row['source_text'])
    # Update the maximum length if the current row length is greater
    max_length=max(max_length,row_length)

  print("Max length in dataset ", max_length)
  return max_length

In [ ]:
def collate_fn(batch):
  sources, target=zip(*batch)
  padded_sources=pad_sequence(sources, batch_first=True)
  padded_targets=pad_sequence(targets, batch_first=True)
  return padded_sources, padded_targets

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, embedding_dim, num_heads):
    super(MultiHeadAttention,self).__init__()
    assert embedding_dim%num_heads==0, "embedding_dim must be divisible by num_heads"
    self.embedding_dim=embedding_dim
    self.num_heads=num_heads
    self.dim_perhead=embedding_dim//num_heads
    self.W_q=nn.Linear(embedding_dim, embedding_dim)
    self.W_k=nn.Linear(embedding_dim, embedding_dim)
    self.W_v=nn.Linear(embedding_dim, embedding_dim)
    self.W_o=nn.Linear(embedding_dim, embedding_dim)
  def scaled_dot_product_attention(self,Q,K,V, mask=None):
    # Q,K,V Shape: [Batch Size X Num_Heads X Seq_len X Dim per head]
    K=K.transpose(-2,-1)
    attn_scores=torch.matmul(Q,K)/math.sqrt(self.dim_perhead)
    if mask is not None:
      attn_scores=attn_scores.masked_fill(mask==0, -1e9)
    attn_probs=torch.softmax(attn_scores, dim=-1)
    # attn_probs Shape:
    output=torch.matmul(attn_probs, V)
    return output
  def split_heads(self, X):
    batch_size, seq_length, d_model=x.size()
    X=X.view(batch_size, seq_length, self.num_heads, self.dim_perhead)

    x=x.transpose(1,2)
    return x

  def combine_heads(self, x):
    batch_size, _ , seq_length, dim_perhead=x.size()
    x=x.transpose(1,2).contiguous()
    x=x.view(batch_size, seq_length, self.embedding_dim)
    return x

  def forward(self, Q, K, V, mask=None):
    Q=self.split_heads(self.W_q(Q))
    K=self.split_heads(self.W_k(K))
    V=self.split_heads(self.W_v(V))

    attn_output=self.scaled_dot_product_attention(Q,K,V,mask)

    output=self.W_o(self.combine_heads(attn_output))
    return output


In [43]:
class PositionWiseForward(nn.Module):
  def __init__(self, d_model, d_ff):
    super(PositionWiseFeedForward,self).__init__()
    self.fc1=nn.Linear(d_model,d_ff)
    self.fc2=nn.Linear(d_ff, d_model)
  def forward(self, x):
    return self.fc2(F.relu(self.fc1(x)))

class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_seq_length):
    super(PositionalEncoding,self).__init__()
    pe=torch.zeros(max_seq_length,d_model)
    position=torch.arange(0,max_seq_length,dtype=torch.float).unsqueeze()
    div_term=torch.exp(torch.arange(0,d_model,2).float()*-(math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(position*div_term)
    pos[:,1::2]=torch.cos(position*div_term)
    self.register_buffer('pe',pe.unsqueeze(0))
  def forward(self,x):
    return x+self.pe[:,:x.size(1)]

In [ ]:
class EncoderLayer(nn.Module):
  def __init__(self,d_model,num_heads, d_ff, dropout):
    super(EncoderLayer,self).__init__()
    self.self_attn=MultiHeadAttention(d_model,num_heads)
    self.feed_forward=PositionWiseFeedForward(d_model,d_ff)
    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(dropout)

  def forward(self,x,mask):
    attn_output=self.self_attn(x,x,x,mask)
    x=self.norm1(x+self.dropout(attn_output))
    ff_output=self.feed_forward(x)

    x=self.norm2(x+self.dropout(ff_output))
    return x


class DecoderLayer(nn.Module):
  def __init__(self,d_model, num_heads, d_ff, dropout):
    super(DecoderLayer,self)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')